In [0]:
%run "/Workspace/Users/ashish.bv.singh@accenture.com/__init__"

In [0]:
sourcepath = "/Volumes/scd/source/sourcefiles"
schemapath = "/Volumes/scd/metadata/schemas/"
checkpointpath = "/Volumes/scd/metadata/checkpoints/"

In [0]:
df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", schemapath)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("cloudFiles.maxFilesPerTrigger", 1)
    .load(sourcepath)
)

In [0]:
# df = spark.read.csv("/Volumes/scd/source/sourcefiles/", header=True, inferSchema=True)

In [0]:
# df.count()

In [0]:
# This code is for legacy  without autoloader
# type1_table_name = "scd.type1.target"
# if not spark.catalog.tableExists(type1_table_name):
#     print(f"Performing full load for {type1_table_name}")
#     df.write.format("delta").saveAsTable(f"{type1_table_name}")
# else: 
#     print(f"Performing SCD1 load for {type1_table_name}")
#     target_delta_table = DeltaTable.forName(spark, type1_table_name)
#     target_delta_table.alias('t').merge(
#         df.alias('s'),
#         "t.customer_id = s.customer_id")\
#         .whenMatchedUpdateAll()\
#         .whenNotMatchedInsertAll()\
#         .execute()
    

In [0]:
from delta.tables import DeltaTable

type1_table_name = "scd.type1.target"

def upsert_to_delta(microBatchDF, batchId):

    if not spark.catalog.tableExists(type1_table_name):

        print(f"First Run - Full Load for {type1_table_name}")

        (
            microBatchDF.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(type1_table_name)
        )

    else:

        print(f"Merge Run - SCD Type 1 for {type1_table_name}")

        target_delta_table = DeltaTable.forName(
            spark,
            type1_table_name
        )

        (
            target_delta_table.alias("t")
            .merge(
                microBatchDF.alias("s"),
                "t.customer_id = s.customer_id"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )


query = (
    df.writeStream
    .foreachBatch(upsert_to_delta)
    .option("checkpointLocation", checkpointpath)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()